## End-to-End Structural Feature Prediction

This notebook predicts phase labels and structural descriptors for screening compounds using trained machine-learning models.

Workflow:
1. Predict the phase label.
2. Predict first- and second-phase structural descriptors.
3. Merge all predictions into one result file.

The final results are saved in the `predict` directory.

## label

In [1]:
import os
import joblib
import numpy as np
import pandas as pd


# File paths
SCREENING_PATH = "data/compound_new_MCs.csv"
TRAIN_PATH = "data/end_to_end_split/structure/label_train_pool.csv"

SCALER_PATH = "model/end to end/label_scaler.pkl"
TOP_IDX_PATH = "model/end to end/label_top_idx.npy"
MODEL_PATH = "model/end to end/xgb_classifier.pkl"

SAVE_PATH = "predict/predicted_label_results.csv"


# Load screening data
df = pd.read_csv(SCREENING_PATH)


# Get the exact feature order used during training
train_data = pd.read_csv(TRAIN_PATH)

feature_cols = [
    col for col in train_data.columns
    if col not in ["compound", "label"]
]

missing_cols = [
    col for col in feature_cols
    if col not in df.columns
]

if missing_cols:
    raise KeyError(
        f"Missing feature columns: {missing_cols}"
    )


# Prepare features
X_new = df[feature_cols]

scaler = joblib.load(SCALER_PATH)
X_new_scaled = scaler.transform(X_new)


# Select the top features used by the classifier
top_idx = np.load(
    TOP_IDX_PATH,
    allow_pickle=True
)

X_new_selected = X_new_scaled[:, top_idx]


# Load classifier and predict
clf = joblib.load(MODEL_PATH)

threshold = 0.65

y_proba_new = clf.predict_proba(
    X_new_selected
)[:, 1]

y_pred_new = (
    y_proba_new >= threshold
).astype(int)


# Save predictions
compound_col = (
    "compound" if "compound" in df.columns
    else "compounds"
)

save_df = pd.DataFrame({
    "compound": df[compound_col],
    "label_probability": y_proba_new,
    "label": y_pred_new
})

os.makedirs("predict", exist_ok=True)

save_df.to_csv(
    SAVE_PATH,
    index=False
)

print(save_df["label"].value_counts())
print(f"\nPredictions saved to: {SAVE_PATH}")

0    15499
1     9513
Name: label, dtype: int64

Predictions saved to: predict/predicted_label_results.csv


## Grain1

In [2]:
import os
import numpy as np
import pandas as pd
import joblib

os.makedirs("predict", exist_ok=True)

# Load model and selected features
model = joblib.load("model/end to end/xgb_Grain1.pkl")

selected_features = np.load(
    "model/end to end/Grain1_selected_features.npy",
    allow_pickle=True
).tolist()

# Load screening data
df_new = pd.read_csv(
    "data/compound_new_MCs.csv"
)

# Check selected features
missing_features = [
    feature for feature in selected_features
    if feature not in df_new.columns
]

if missing_features:
    raise KeyError(
        f"Missing features: {missing_features}"
    )

X_new = df_new[selected_features]

# Predict log-transformed Grain1
y_pred_log = model.predict(X_new)

# Convert back to the original Grain1 scale
y_pred = np.expm1(y_pred_log)

# Save results
out = pd.DataFrame({
    "compound": df_new["compound"],
    "Grain_1": y_pred
})

out.to_csv(
    "predict/predicted_grain1_results.csv",
    index=False
)

print(out.head())
print("[Saved] predict/predicted_grain1_results.csv")

        compound    Grain_1
0  Ag0.95Ce0.05N  16.261442
1    Ag0.9Ce0.1N  14.941677
2    Ag0.8Ce0.2N  17.444633
3    Ag0.5Ce0.5N  17.554060
4  Ag0.95Co0.05N  15.697699
[Saved] predict/predicted_grain1_results.csv


## a1

In [1]:
import os
import numpy as np
import pandas as pd
import joblib

os.makedirs("predict", exist_ok=True)

# Load model and selected features
model = joblib.load("model/end to end/xgb_a1.pkl")

selected_features = np.load(
    "model/end to end/a1_selected_features.npy",
    allow_pickle=True
).tolist()

# Load screening data
df_new = pd.read_csv(
    "data/compound_new_MCs.csv"
)

# Check selected features
missing_features = [
    feature for feature in selected_features
    if feature not in df_new.columns
]

if missing_features:
    raise KeyError(
        f"Missing features: {missing_features}"
    )

X_new = df_new[selected_features]

# Predict a1
y_pred = model.predict(X_new)

# Save results
out = pd.DataFrame({
    "compound": df_new["compound"],
    "a_1": y_pred
})

out.to_csv(
    "predict/predicted_a1_results.csv",
    index=False
)

print(out.head())
print("[Saved] predict/predicted_a1_results.csv")

        compound       a_1
0  Ag0.95Ce0.05N  5.095881
1    Ag0.9Ce0.1N  4.648705
2    Ag0.8Ce0.2N  5.749419
3    Ag0.5Ce0.5N  3.987883
4  Ag0.95Co0.05N  4.792680
[Saved] predict/predicted_a1_results.csv


## b1

In [1]:
import os
import numpy as np
import pandas as pd
import joblib

os.makedirs("predict", exist_ok=True)

# Load model and selected features
model = joblib.load("model/end to end/xgb_b1.pkl")

selected_features = np.load(
    "model/end to end/b1_selected_features.npy",
    allow_pickle=True
).tolist()

# Load screening data
df_new = pd.read_csv(
    "data/compound_new_MCs.csv"
)

# Check selected features
missing_features = [
    feature for feature in selected_features
    if feature not in df_new.columns
]

if missing_features:
    raise KeyError(
        f"Missing features: {missing_features}"
    )

X_new = df_new[selected_features]

# Predict a1
y_pred = model.predict(X_new)

# Save results
out = pd.DataFrame({
    "compound": df_new["compound"],
    "b_1": y_pred
})

out.to_csv(
    "predict/predicted_b1_results.csv",
    index=False
)

print(out.head())
print("[Saved] predict/predicted_b1_results.csv")

        compound       b_1
0  Ag0.95Ce0.05N  4.471525
1    Ag0.9Ce0.1N  4.695375
2    Ag0.8Ce0.2N  6.243092
3    Ag0.5Ce0.5N  5.142775
4  Ag0.95Co0.05N  3.828983
[Saved] predict/predicted_b1_results.csv


## c1

In [1]:
import os
import numpy as np
import pandas as pd
import joblib

os.makedirs("predict", exist_ok=True)

# Load model and selected features
model = joblib.load("model/end to end/xgb_c1.pkl")

selected_features = np.load(
    "model/end to end/c1_selected_features.npy",
    allow_pickle=True
).tolist()

# Load screening data
df_new = pd.read_csv(
    "data/compound_new_MCs.csv"
)

# Check selected features
missing_features = [
    feature for feature in selected_features
    if feature not in df_new.columns
]

if missing_features:
    raise KeyError(
        f"Missing features: {missing_features}"
    )

X_new = df_new[selected_features]

# Predict a1
y_pred = model.predict(X_new)

# Save results
out = pd.DataFrame({
    "compound": df_new["compound"],
    "c_1": y_pred
})

out.to_csv(
    "predict/predicted_c1_results.csv",
    index=False
)

print(out.head())
print("[Saved] predict/predicted_c1_results.csv")

        compound       c_1
0  Ag0.95Ce0.05N  4.644791
1    Ag0.9Ce0.1N  4.858928
2    Ag0.8Ce0.2N  5.063380
3    Ag0.5Ce0.5N  4.754720
4  Ag0.95Co0.05N  4.600904
[Saved] predict/predicted_c1_results.csv


## grain2

In [2]:
import os
import numpy as np
import pandas as pd
import joblib

os.makedirs("predict", exist_ok=True)

# Load model and selected features
model = joblib.load("model/end to end/xgb_Grain2.pkl")

selected_features = np.load(
    "model/end to end/Grain2_selected_features.npy",
    allow_pickle=True
).tolist()

# Load screening data
df_new = pd.read_csv(
    "data/compound_new_MCs_2.csv"
)

# Check selected features
missing_features = [
    feature for feature in selected_features
    if feature not in df_new.columns
]

if missing_features:
    raise KeyError(
        f"Missing features: {missing_features}"
    )

X_new = df_new[selected_features]

# Predict log-transformed Grain1
y_pred_log = model.predict(X_new)

# Convert back to the original Grain1 scale
y_pred = np.expm1(y_pred_log)

# Save results
out = pd.DataFrame({
    "compound": df_new["compound"],
    "Grain_2": y_pred
})

out.to_csv(
    "predict/predicted_grain2_results.csv",
    index=False
)

print(out.head())
print("[Saved] predict/predicted_grain2_results.csv")

      compound    Grain_2
0  Ag0.5Cr0.5N  34.388611
1  Ag0.5Mn0.5N  29.372099
2  Ag0.5Ni0.5N  34.388611
3  Ag0.5Pd0.5N  46.638672
4  Ag0.5Ti0.5N  29.372099
[Saved] predict/predicted_grain2_results.csv


## a2

In [1]:
import os
import numpy as np
import pandas as pd
import joblib

os.makedirs("predict", exist_ok=True)

# Load model and selected features
model = joblib.load("model/end to end/xgb_a2.pkl")

selected_features = np.load(
    "model/end to end/a2_selected_features.npy",
    allow_pickle=True
).tolist()

# Load screening data
df_new = pd.read_csv(
    "data/compound_new_MCs_2.csv"
)

# Check selected features
missing_features = [
    feature for feature in selected_features
    if feature not in df_new.columns
]

if missing_features:
    raise KeyError(
        f"Missing features: {missing_features}"
    )

X_new = df_new[selected_features]

# Predict a1
y_pred = model.predict(X_new)

# Save results
out = pd.DataFrame({
    "compound": df_new["compound"],
    "a_2": y_pred
})

out.to_csv(
    "predict/predicted_a2_results.csv",
    index=False
)

print(out.head())
print("[Saved] predict/predicted_a2_results.csv")

      compound       a_2
0  Ag0.5Cr0.5N  5.158931
1  Ag0.5Mn0.5N  4.784737
2  Ag0.5Ni0.5N  3.037118
3  Ag0.5Pd0.5N  4.282328
4  Ag0.5Ti0.5N  5.285024
[Saved] predict/predicted_a2_results.csv


## b2

In [1]:
import os
import numpy as np
import pandas as pd
import joblib

os.makedirs("predict", exist_ok=True)

# Load model and selected features
model = joblib.load("model/end to end/xgb_b2.pkl")

selected_features = np.load(
    "model/end to end/b2_selected_features.npy",
    allow_pickle=True
).tolist()

# Load screening data
df_new = pd.read_csv(
    "data/compound_new_MCs_2.csv"
)

# Check selected features
missing_features = [
    feature for feature in selected_features
    if feature not in df_new.columns
]

if missing_features:
    raise KeyError(
        f"Missing features: {missing_features}"
    )

X_new = df_new[selected_features]

# Predict a1
y_pred = model.predict(X_new)

# Save results
out = pd.DataFrame({
    "compound": df_new["compound"],
    "b_2": y_pred
})

out.to_csv(
    "predict/predicted_b2_results.csv",
    index=False
)

print(out.head())
print("[Saved] predict/predicted_b2_results.csv")

      compound       b_2
0  Ag0.5Cr0.5N  5.612646
1  Ag0.5Mn0.5N  5.370048
2  Ag0.5Ni0.5N  4.893002
3  Ag0.5Pd0.5N  5.434428
4  Ag0.5Ti0.5N  5.662207
[Saved] predict/predicted_b2_results.csv


## c2

In [2]:
import os
import numpy as np
import pandas as pd
import joblib

os.makedirs("predict", exist_ok=True)

# Load model and selected features
model = joblib.load("model/end to end/xgb_c2.pkl")

selected_features = np.load(
    "model/end to end/c2_selected_features.npy",
    allow_pickle=True
).tolist()

# Load screening data
df_new = pd.read_csv(
    "data/compound_new_MCs_2.csv"
)

# Check selected features
missing_features = [
    feature for feature in selected_features
    if feature not in df_new.columns
]

if missing_features:
    raise KeyError(
        f"Missing features: {missing_features}"
    )

X_new = df_new[selected_features]

# Predict a1
y_pred = model.predict(X_new)

# Save results
out = pd.DataFrame({
    "compound": df_new["compound"],
    "c_2": y_pred
})

out.to_csv(
    "predict/predicted_c2_results.csv",
    index=False
)

print(out.head())
print("[Saved] predict/predicted_c2_results.csv")

      compound        c_2
0  Ag0.5Cr0.5N   4.266585
1  Ag0.5Mn0.5N  10.585428
2  Ag0.5Ni0.5N   5.977458
3  Ag0.5Pd0.5N   7.640924
4  Ag0.5Ti0.5N   8.426192
[Saved] predict/predicted_c2_results.csv


In [3]:
import os
import pandas as pd


predict_dir = "predict"

label_path = os.path.join(
    predict_dir,
    "predicted_label_results.csv"
)

result = pd.read_csv(label_path)


# ============================================================
# 1. Basic checks
# ============================================================

required_label_cols = [
    "compound",
    "label"
]

missing_cols = [
    col for col in required_label_cols
    if col not in result.columns
]

if missing_cols:
    raise KeyError(
        f"Missing columns in label file: {missing_cols}"
    )

if result["compound"].duplicated().any():
    duplicated = result.loc[
        result["compound"].duplicated(False),
        "compound"
    ].tolist()

    raise ValueError(
        f"Duplicated compounds in label file: {duplicated[:10]}"
    )

result["label"] = result["label"].astype(int)


# ============================================================
# 2. Prediction files
# ============================================================

prediction_files = {
    "predicted_grain1_results.csv": "Grain_1",
    "predicted_a1_results.csv": "a_1",
    "predicted_b1_results.csv": "b_1",
    "predicted_c1_results.csv": "c_1",
    "predicted_grain2_results.csv": "Grain_2",
    "predicted_a2_results.csv": "a_2",
    "predicted_b2_results.csv": "b_2",
    "predicted_c2_results.csv": "c_2"
}


# ============================================================
# 3. Merge predictions
# ============================================================

for file_name, value_col in prediction_files.items():

    file_path = os.path.join(
        predict_dir,
        file_name
    )

    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"Prediction file not found: {file_name}"
        )

    temp = pd.read_csv(file_path)

    required_cols = [
        "compound",
        value_col
    ]

    missing_cols = [
        col for col in required_cols
        if col not in temp.columns
    ]

    if missing_cols:
        raise KeyError(
            f"Missing columns in {file_name}: {missing_cols}"
        )

    if temp["compound"].duplicated().any():
        duplicated = temp.loc[
            temp["compound"].duplicated(False),
            "compound"
        ].tolist()

        raise ValueError(
            f"Duplicated compounds in {file_name}: "
            f"{duplicated[:10]}"
        )

    result = result.merge(
        temp[["compound", value_col]],
        on="compound",
        how="left",
        validate="one_to_one"
    )


# ============================================================
# 4. Fill second-phase descriptors
# ============================================================

first_phase_cols = [
    "Grain_1",
    "a_1",
    "b_1",
    "c_1"
]

second_phase_cols = [
    "Grain_2",
    "a_2",
    "b_2",
    "c_2"
]

result.loc[
    result["label"] == 0,
    second_phase_cols
] = 0.0


# ============================================================
# 5. Check missing predictions
# ============================================================

first_phase_missing = result[
    first_phase_cols
].isna().any(axis=1)

if first_phase_missing.any():

    missing_compounds = result.loc[
        first_phase_missing,
        "compound"
    ].tolist()

    raise ValueError(
        "Missing first-phase predictions for: "
        f"{missing_compounds[:10]}"
    )


second_phase_missing = (
    (result["label"] == 1)
    & result[second_phase_cols].isna().any(axis=1)
)

if second_phase_missing.any():

    missing_compounds = result.loc[
        second_phase_missing,
        "compound"
    ].tolist()

    raise ValueError(
        "Missing second-phase predictions for "
        "label=1 compounds: "
        f"{missing_compounds[:10]}"
    )


# ============================================================
# 6. Save results
# ============================================================

save_path = os.path.join(
    predict_dir,
    "all_structural_prediction_results.csv"
)

result.to_csv(
    save_path,
    index=False
)

print("Total compounds:", len(result))
print("Single-phase compounds:", (result["label"] == 0).sum())
print("Two-phase compounds:", (result["label"] == 1).sum())
print("Remaining missing values:")
print(result.isna().sum())

print(f"[Saved] {save_path}")

display(result.head())

Total compounds: 25012
Single-phase compounds: 15499
Two-phase compounds: 9513
Remaining missing values:
compound             0
label_probability    0
label                0
Grain_1              0
a_1                  0
b_1                  0
c_1                  0
Grain_2              0
a_2                  0
b_2                  0
c_2                  0
dtype: int64
[Saved] predict\all_structural_prediction_results.csv


,compound,label_probability,label,Grain_1,a_1,b_1,c_1,Grain_2,a_2,b_2,c_2
0,Ag0.95Ce0.05N,0.279003,0,16.261442,5.095880,4.471525,4.644791,0.0,0.0,0.0,0.0
1,Ag0.9Ce0.1N,0.279003,0,14.941677,4.648705,4.695375,4.858928,0.0,0.0,0.0,0.0
2,Ag0.8Ce0.2N,0.279003,0,17.444633,5.749419,6.243092,5.063380,0.0,0.0,0.0,0.0
3,Ag0.5Ce0.5N,0.521989,0,17.554060,3.987883,5.142775,4.754720,0.0,0.0,0.0,0.0
4,Ag0.95Co0.05N,0.277546,0,15.697699,4.792680,3.828983,4.600904,0.0,0.0,0.0,0.0
